In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os

# Configure plotting and data routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SimplePassword1"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all data from MongoDB
df = pd.DataFrame.from_records(db.read({}))

# Drop _id if present
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(df.columns)
# print(df.head())


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),

    # Logo and unique identifier
    html.Div([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height':'100px'}),
        html.P("Developer: Jake Whaley")  # unique identifier
    ], style={'textAlign':'center'}),

    html.Hr(),

    # Interactive filtering options
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label':'Water Rescue', 'value':'water'},
                {'label':'Mountain/Wilderness Rescue', 'value':'mountain'},
                {'label':'Disaster/Individual Tracking', 'value':'disaster'},
                {'label':'Reset', 'value':'reset'}
            ],
            value='reset',
            labelStyle={'display':'inline-block', 'margin-right':'10px'}
        )
    ),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),

    html.Br(),
    html.Hr(),

    # Charts and geolocation map side by side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter data table based on interactive options
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'reset' or filter_type is None:
        df_filtered = pd.DataFrame.from_records(db.read({}))
    elif filter_type == 'water':
        df_filtered = pd.DataFrame.from_records(db.read({"breed": {"$in":["Labrador Retriever","Chesapeake Bay Retriever","Newfoundland"]}}))
    elif filter_type == 'mountain':
        df_filtered = pd.DataFrame.from_records(db.read({"breed": {"$in":["German Shepherd","Border Collie","Australian Shepherd"]}}))
    elif filter_type == 'disaster':
        df_filtered = pd.DataFrame.from_records(db.read({"breed": {"$in":["Bloodhound","Beagle"]}}))
    else:
        df_filtered = pd.DataFrame.from_records(db.read({}))

    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
        
    return df_filtered.to_dict('records')


# Update pie chart based on current table data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' not in dff.columns:
        return []  # No pie chart if column missing
    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Distribution of Breeds')
        )
    ]


# Highlight selected columns in data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update geolocation chart for selected data row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []  # Nothing to display

    dff = pd.DataFrame.from_dict(viewData)
    
    # Default to first row
    row = index[0] if index else 0

    # Check that latitude/longitude columns exist
    if len(dff.columns) < 15:  # columns 13,14 used for lat/lon
        return []

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row,13], dff.iloc[row,14]],  # Lat, Lon
                    children=[
                        dl.Tooltip(str(dff.iloc[row,4])),  # Breed
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row,9]))  # Name
                        ])
                    ]
                )
            ]
        )
    ]

#----------------------
# Run the server (Codio-ready)
#----------------------
from IPython.display import display, HTML

codio_url = "https://extendbread-tradesilver-8050.codio.io"
display(HTML(f'<a href="{codio_url}" target="_blank">Click here to open the Dashboard</a>'))

app.run_server(debug=True, host='0.0.0.0', port=8050, mode='external')

Dash app running on http://0.0.0.0:8050/
